In [24]:
# Load the matrix from the CSV file
matrix = pd.read_csv('Ergebnisse/LP_Verständlichkeit.csv', header=None)
dozenten = pd.read_csv('CSVs/dozents.csv')

print(matrix.columns)


print(matrix)
print(dozenten)

Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
        96,  97,  98,  99, 100, 101, 102, 103, 104, 105],
      dtype='int64', length=106)
     0    1    2    3    4    5    6    7    8    9    ...  96   97   98   \
0      0    0    0    0    4    0    0    1    0    2  ...    1    2    1   
1      1    0    0    1    0    1    1    3    0    0  ...    3    0    1   
2      2    0    0    0    2    0    0    1    1    1  ...    0    1    1   
3      1    1    1    0    0    1    1    2    1    2  ...    0    1    0   
4      0    0    1    0    0    1    3    1    0    0  ...    0    0    1   
..   ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
101    0    2    0    0    0    1    0    1    1    0  ...    0    1    1   
102    1    0    0    1    1    2    0    1    0    0  ...    0    0    1   
103    0    1    3    0    0    0    1    0    0    1  ...    0    0    1   
104    2    0    1    0    0    0    1    1    2    2  ...    0    0   

In [25]:
# Annahme: Der Index von `dozenten` enthält die IDs, die mit Namen assoziiert sind
#dozenten.columns = ['lastName']

# Mapping von IDs zu Namen erstellen
namen_mapping = dozenten['lastName'].to_dict()

# Überprüfen, welche Spalte die IDs enthält
id_column = 0  # Die Spalte, die die IDs enthält (anpassen falls nötig)

# Setze die Namen als Index (y-Achse)
matrix.index = matrix[id_column].map(namen_mapping)

# Setze die Namen als Spaltenbeschriftungen (x-Achse)
matrix.columns = [namen_mapping.get(col, col) for col in range(matrix.shape[1])]

# Ergebnis anzeigen
print(matrix)

# Ergebnis anzeigen
print(matrix)

           Mitrevski  Asteroth  Kees  Rieke  Küstenmacher  Hense  Hackelöer  \
0                                                                             
Mitrevski          0         0     0      0             4      0          0   
Asteroth           1         0     0      1             0      1          1   
Kees               2         0     0      0             2      0          0   
Asteroth           1         1     1      0             0      1          1   
Mitrevski          0         0     1      0             0      1          3   
...              ...       ...   ...    ...           ...    ...        ...   
Mitrevski          0         2     0      0             0      1          0   
Asteroth           1         0     0      1             1      2          0   
Mitrevski          0         1     3      0             0      0          1   
Kees               2         0     1      0             0      0          1   
Asteroth           1         0     0      0         

In [26]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.projections")

# Load the matrix from the CSV file
#matrix = pd.read_csv('Ergebnisse/LP_Verständlichkeit.csv', header=None)
#dozenten = pd.read_csv('CSVs/dozents.csv')

# Ensure it's a square matrix
if matrix.shape[0] != matrix.shape[1]:
    print(matrix)
    #raise ValueError("The matrix must be square (m*m).")

# Create a directed graph from the adjacency matrix
G = nx.from_pandas_adjacency(matrix, create_using=nx.DiGraph)

# Function to filter edges and nodes based on a threshold
def plot_filtered_graph(threshold):
    filtered_G = nx.DiGraph()
    
    # Add edges that meet the threshold
    for u, v, data in G.edges(data=True):
        weight = matrix.loc[u, v]
        if weight >= threshold:
            filtered_G.add_edge(u, v, weight=weight)
    
    # Remove isolated nodes (nodes with no edges)
    isolated_nodes = list(nx.isolates(filtered_G))
    filtered_G.remove_nodes_from(isolated_nodes)
    
    # Plot the filtered graph
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(filtered_G)
    nx.draw(filtered_G, pos, with_labels=True, node_color='skyblue', node_size=2000, font_size=10, font_color='black', font_weight='bold', edge_color='gray', arrowsize=20)
    edge_labels = nx.get_edge_attributes(filtered_G, 'weight')
    nx.draw_networkx_edge_labels(filtered_G, pos, edge_labels=edge_labels)
    plt.title(f'Graph with threshold >= {threshold}')
    plt.show()

# Interactive slider for threshold selection
max_weight = matrix.max().max()
interact(plot_filtered_graph, threshold=IntSlider(min=0, max=int(max_weight), step=1, value=2))


interactive(children=(IntSlider(value=2, description='threshold', max=6), Output()), _dom_classes=('widget-int…

<function __main__.plot_filtered_graph(threshold)>

In [27]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.projections")

# Load the matrix from the CSV file
# matrix = pd.read_csv('Ergebnisse/LP_Verständlichkeit.csv', header=None)
# dozenten = pd.read_csv('CSVs/dozents.csv')

# Ensure it's a square matrix
if matrix.shape[0] != matrix.shape[1]:
    print(matrix)
    raise ValueError("The matrix must be square (m*m).")

# Rename matrix index and columns to match node labels
matrix.index = matrix.columns = list(range(matrix.shape[0]))

# Create a directed graph from the adjacency matrix
G = nx.from_pandas_adjacency(matrix, create_using=nx.DiGraph)

# Function to filter edges and nodes based on a threshold
def plot_filtered_graph(threshold):
    filtered_G = nx.DiGraph()
    
    # Add edges that meet the threshold
    for u, v, data in G.edges(data=True):
        try:
            weight = matrix.at[u, v]  # Use `at` for single-value access
        except KeyError:
            continue  # Skip if the edge is not in the matrix
        
        if weight >= threshold:
            filtered_G.add_edge(u, v, weight=weight)
    
    # Remove isolated nodes (nodes with no edges)
    isolated_nodes = list(nx.isolates(filtered_G))
    filtered_G.remove_nodes_from(isolated_nodes)
    
    # Plot the filtered graph
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(filtered_G)
    nx.draw(filtered_G, pos, with_labels=True, node_color='skyblue', node_size=2000, 
            font_size=10, font_color='black', font_weight='bold', edge_color='gray', arrowsize=20)
    edge_labels = nx.get_edge_attributes(filtered_G, 'weight')
    nx.draw_networkx_edge_labels(filtered_G, pos, edge_labels=edge_labels)
    plt.title(f'Graph with threshold >= {threshold}')
    plt.show()

# Interactive slider for threshold selection
max_weight = matrix.max().max()
interact(plot_filtered_graph, threshold=IntSlider(min=0, max=int(max_weight), step=1, value=2))


interactive(children=(IntSlider(value=2, description='threshold', max=6), Output()), _dom_classes=('widget-int…

<function __main__.plot_filtered_graph(threshold)>

In [5]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.projections")

# Load the matrix from the CSV file without column names
matrix = pd.read_csv('Ergebnisse/LP_Verständlichkeit.csv', header=None)

# Ensure it's a square matrix
if matrix.shape[0] != matrix.shape[1]:
    raise ValueError("The matrix must be square (m*m).")

# Assign default indices and column names if missing
matrix.index = range(matrix.shape[0])
matrix.columns = range(matrix.shape[1])

# Create a directed graph from the adjacency matrix
G = nx.from_pandas_adjacency(matrix, create_using=nx.DiGraph)

# Function to filter edges and nodes based on a threshold
def plot_filtered_graph(threshold):
    filtered_G = nx.DiGraph()
    
    # Add edges that meet the threshold
    for u, v, data in G.edges(data=True):
        weight = matrix.loc[u, v]
        if weight >= threshold:
            filtered_G.add_edge(u, v, weight=weight)
    
    # Remove isolated nodes (nodes with no edges)
    isolated_nodes = list(nx.isolates(filtered_G))
    filtered_G.remove_nodes_from(isolated_nodes)
    
    # Ensure graph layout has upward orientation
    pos = nx.spring_layout(filtered_G)
    pos = {node: (x, y * -1) for node, (x, y) in pos.items()}  # Invert y-axis for upward orientation
    
    # Plot the filtered graph
    plt.figure(figsize=(10, 8))
    nx.draw(filtered_G, pos, with_labels=True, node_color='skyblue', node_size=2000, font_size=10, font_color='black', font_weight='bold', edge_color='gray')
    edge_labels = nx.get_edge_attributes(filtered_G, 'weight')
    nx.draw_networkx_edge_labels(filtered_G, pos, edge_labels=edge_labels)
    plt.title(f'Graph with threshold >= {threshold}')
    plt.show()

# Interactive slider for threshold selection
max_weight = matrix.max().max()
interact(plot_filtered_graph, threshold=IntSlider(min=0, max=int(max_weight), step=1, value=4))


interactive(children=(IntSlider(value=0, description='threshold', max=6), Output()), _dom_classes=('widget-int…

<function __main__.plot_filtered_graph(threshold)>

In [9]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.projections")

# Load the matrix from the CSV file without column names
matrix = pd.read_csv('Ergebnisse/LP_Verständlichkeit.csv', header=None)

# Ensure it's a square matrix
if matrix.shape[0] != matrix.shape[1]:
    raise ValueError("The matrix must be square (m*m).")

# Assign default indices and column names if missing
matrix.index = range(matrix.shape[0])
matrix.columns = range(matrix.shape[1])

# Create a directed graph from the adjacency matrix
G = nx.from_pandas_adjacency(matrix, create_using=nx.DiGraph)

# Precompute layout for faster plotting
pos = nx.spring_layout(G, seed=42)
pos = {node: (x, y * -1) for node, (x, y) in pos.items()}  # Invert y-axis for upward orientation

# Function to filter edges and nodes based on a threshold
def plot_filtered_graph(threshold):
    filtered_edges = [(u, v) for u, v, data in G.edges(data=True) if matrix.loc[u, v] >= threshold]
    filtered_nodes = set(u for u, v in filtered_edges).union(v for u, v in filtered_edges)
    
    filtered_G = G.edge_subgraph(filtered_edges).copy()
    
    # Remove isolated nodes
    isolated_nodes = set(filtered_G.nodes) - filtered_nodes
    filtered_G.remove_nodes_from(isolated_nodes)
    
    # Plot the filtered graph
    plt.figure(figsize=(10, 8))
    nx.draw(filtered_G, pos, with_labels=True, node_color='skyblue', node_size=2000, font_size=10, font_color='black', font_weight='bold', edge_color='gray')
    edge_labels = nx.get_edge_attributes(filtered_G, 'weight')
    nx.draw_networkx_edge_labels(filtered_G, pos, edge_labels=edge_labels)
    plt.title(f'Graph with threshold >= {threshold}')
    plt.show()

# Interactive slider for threshold selection
max_weight = matrix.max().max()
interact(plot_filtered_graph, threshold=IntSlider(min=0, max=int(max_weight), step=1, value=4))

interactive(children=(IntSlider(value=4, description='threshold', max=6), Output()), _dom_classes=('widget-int…

<function __main__.plot_filtered_graph(threshold)>

In [8]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.projections")

# Ensure it's a square matrix
if matrix.shape[0] != matrix.shape[1]:
    raise ValueError("The matrix must be square (m*m).")

# Assign default indices and column names if missing
matrix.index = range(matrix.shape[0])
matrix.columns = range(matrix.shape[1])

# Create a directed graph from the adjacency matrix
G = nx.from_pandas_adjacency(matrix, create_using=nx.DiGraph)

# Precompute a hierarchical layout for upward direction using topological sort
try:
    hierarchy = list(nx.topological_sort(G))
    pos = {node: (0, -i) for i, node in enumerate(hierarchy)}
except nx.NetworkXUnfeasible:
    pos = nx.spring_layout(G, seed=42)
    pos = {node: (x, -y) for node, (x, y) in pos.items()}

# Function to filter edges and nodes based on a threshold
def plot_filtered_graph(threshold):
    filtered_edges = [(u, v) for u, v, data in G.edges(data=True) if matrix.loc[u, v] >= threshold]
    filtered_nodes = set(u for u, v in filtered_edges).union(v for u, v in filtered_edges)
    
    filtered_G = G.edge_subgraph(filtered_edges).copy()
    
    # Remove isolated nodes
    isolated_nodes = set(filtered_G.nodes) - filtered_nodes
    filtered_G.remove_nodes_from(isolated_nodes)
    
    # Plot the filtered graph
    plt.figure(figsize=(10, 8))
    nx.draw(filtered_G, pos, with_labels=True, node_color='skyblue', node_size=2000, font_size=10, font_color='black', font_weight='bold', edge_color='gray')
    edge_labels = nx.get_edge_attributes(filtered_G, 'weight')
    nx.draw_networkx_edge_labels(filtered_G, pos, edge_labels=edge_labels)
    plt.title(f'Graph with threshold >= {threshold}')
    plt.show()

# Interactive slider for threshold selection
max_weight = matrix.max().max()
interact(plot_filtered_graph, threshold=IntSlider(min=0, max=int(max_weight), step=1, value=4))

interactive(children=(IntSlider(value=0, description='threshold', max=6), Output()), _dom_classes=('widget-int…

<function __main__.plot_filtered_graph(threshold)>